In [ ]:
import os
from openai import OpenAI

os.environ["SOY_TOKEN"] = 'y1__'

client = OpenAI(
    api_key=os.getenv("SOY_TOKEN")
)

# Выполните запрос к модели
completion = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": "Hello, world!"
        }
    ]
)

# Выведите результат
print(completion.choices[0].message.content)

APIConnectionError: Connection error.

In [5]:
res = await models.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of the moon?"}
    ]
)
res

APIConnectionError: Connection error.

In [ ]:
import time
import requests
import traceback
import json

API_KEY = "<API_key_value>"
FOLDER_ID = "b1goicf28fgnacb1huk5"
FILE_PATH = "...."

def pretty_api_error(response):
    try:
        data = response.json()
    except Exception:
        return response.text

    if "error" in data:
        err = data["error"]
        code = err.get("code", "unknown")
        msg = err.get("message", "no message")
        details = err.get("details", [])
        return f"API Error [{code}]: {msg}\nDetails: {json.dumps(details, ensure_ascii=False, indent=2)}"
    return json.dumps(data, ensure_ascii=False, indent=2)

def main():
    file_uri = f"https://storage.yandexcloud.net/{FILE_PATH}"

    request_data = {
        "uri": file_uri,
        "recognitionModel": {
            "model": "general",
            "audioFormat": {
                "containerAudio": {
                    "containerAudioType": "WAV"
                }
            },
            "languageRestriction": {
                "restrictionType": "WHITELIST",
                "languageCode": ["en-US"]
            },
            "textNormalization": {
                "textNormalization": "TEXT_NORMALIZATION_DISABLED"
            }
        },
        "speakerLabeling": {
            "speakerLabeling": "SPEAKER_LABELING_DISABLED"
        }
    }

    headers = {
        "Authorization": f"Api-key {API_KEY}",
        "x-folder-id": FOLDER_ID
    }

    response = requests.post(
        "https://stt.api.cloud.yandex.net/stt/v3/recognizeFileAsync",
        headers=headers,
        json=request_data,
        verify=False
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Recognition request failed: {response.status_code}\n{pretty_api_error(response)}"
        )

    operation_data = response.json()
    operation_id = operation_data.get("id")
    if not operation_id:
        raise RuntimeError("Operation ID not found in response")

    print(f"Operation ID: {operation_id}")
    print("Waiting for recognition to complete...", end="", flush=True)

    operation_url = f"https://operation.api.cloud.yandex.net/operations/{operation_id}"

    while True:
        op_response = requests.get(operation_url, headers=headers, verify=False)

        if op_response.status_code != 200:
            print(f"\nOperation check failed: {op_response.status_code}")
            print(pretty_api_error(op_response))
            time.sleep(10)
            continue

        op_data = op_response.json()

        if op_data.get("done"):
            if "error" in op_data:
                raise RuntimeError(f"Operation failed:\n{json.dumps(op_data['error'], ensure_ascii=False, indent=2)}")
            break

        print("Waiting for result...")
        time.sleep(10)

    speech_response = requests.get(
        f"https://stt.api.cloud.yandex.net/stt/v3/getRecognition?operation_id={operation_id}",
        headers=headers,
        verify=False
    )

    print(f"\nSpeech Result {speech_response.status_code}")
    if speech_response.status_code != 200:
        raise RuntimeError(
            f"Result request failed: {speech_response.status_code}\n{pretty_api_error(speech_response)}"
        )
    return speech_response.text


if __name__ == "__main__":
    try:
        result = main()
        print("\n\nRecognition Result:")
        print(result)
    except Exception:
        traceback.print_exc()

In [18]:
with open('prompt.txt', 'r') as file:
    prompt = file.read()

print(prompt)

Transcribe this lecture audio into a complete, fully compilable LaTeX document.

Critical requirements:
- Transcribe the lecture as fully and literally as possible.
- Do not summarize, shorten, compress, paraphrase, polish, reinterpret, or rewrite the lecture.
- Preserve the lecture as a lecture, not as a cleaned article, notes, summary, or textbook.
- Do not omit any words, phrases, repetitions, corrections, examples, definitions, intermediate reasoning, or side remarks that are part of the lecture.
- Do not skip any mathematical content under any circumstances.
- Every spoken formula, equation, symbol, variable, index, exponent, operator, sign, Greek letter, condition, interval, inequality, matrix, fraction, derivative, integral, limit, and expression must be preserved and written in valid LaTeX math notation.
- Preserve the order of the content exactly as it appears in the audio.
- Do not merge distant parts of the lecture or reorganize the material for readability.
- Keep the natur

In [ ]:
import base64
import requests
import os
import json


os.environ["SOY_TOKEN"] = 'y1__'

proxy_url = "https://<host>/openai/v1/chat/completions"
proxy_headers = {"Authorization": f"OAuth {os.getenv('SOY_TOKEN')}"}


def wav_to_base64(file_path):
    with open(file_path, 'rb') as wav_file:
        wav_data = wav_file.read()
        base64_encoded = base64.b64encode(wav_data)
        return base64_encoded.decode('utf-8')


file_path = 'output_cut_big.wav'

data = {
  "model": "gpt-4o-audio-preview",
  "modalities": [
    "text"
  ],
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": prompt
        },
        {
          "type": "input_audio",
          "input_audio": {
            "data": wav_to_base64(file_path),
            "format": "wav"
          }
        }
      ]
    }
  ]
}

print('sent')
r = requests.post(
    url=proxy_url,
    data=json.dumps(data, ensure_ascii=False).encode("utf-8"),
    headers=proxy_headers,
    timeout=310,
)

model_response = r.json()["response"]
res = model_response['choices'][0]['message']['content']
with open("result.tex", "w") as f:
    f.write(res)

sent


In [12]:
model_response

{'id': 'chatcmpl-DHxRhuoHOvG3D0ShmdBJraHx9es6r',
 'object': 'chat.completion',
 'created': 1773172457,
 'model': 'gpt-4o-audio-preview-2025-06-03',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '\\documentclass{article}\n\\usepackage[utf8]{inputenc}\n\\usepackage{amsmath}\n\n\\begin{document}\n\n\\title{Lecture Notes}\n\\author{}\n\\date{}\n\n\\maketitle\n\n\\section{Introduction}\n\nThis lecture covers the fundamental concepts of information theory as introduced by \\textit{Hermann Haken}, a prominent scientist in Synergetics.\n\n\\section{Probabilistic Events and Information}\n\nLet us consider two probabilistic events:\n\n\\begin{enumerate}\n    \\item Event A: A professor enters the room.\n    \\item Event B: The professor accuses a student.\n\\end{enumerate}\n\nWe want to determine which of these events provides more information.\n\n\\subsection{Information and Probability}\n\nWe have two probabilistic events. The first event is that a professor ente

In [10]:
! ffmpeg -i Gromov-26.09.wav -ss 1200 -to 1260 -c copy out.wav

/bin/bash: ffmpeg: command not found
